In [ ]:
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install sklearn

In [ ]:
!pip install plotly

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

In [ ]:
import pandas as pd
BASE_DIR = "drive/My Drive/GPT Question Generation and TopDown Learning/"
df = pd.read_csv(BASE_DIR+"ae_generatedqs.csv")
df.head()

In [ ]:
import matplotlib.pyplot as plt

columns_to_include = ["shot_num", "pipeline_verified"]
selected_df = df[columns_to_include]
grouped = selected_df.groupby('shot_num').sum().sort_index()

# Create a histogram
grouped.plot(kind='barh', stacked=True, )
plt.xlabel('Count of Passes')
plt.ylabel('Number of Shots')
plt.title('Pipeline Passes by Number of Shots')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

tax_map = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
label_mapping = {"Remember": 0, "Understand": 1, "Apply": 2, "Analyze": 3, "Evaluate": 4, "Create": 5}
columns_to_include = ["gpt_taxonomy", "pipeline_verified"]
selected_df = df[columns_to_include]
grouped = selected_df.groupby('gpt_taxonomy').sum().sort_index(key=lambda x: x.map(label_mapping))

print(grouped)
# Create a histogram
grouped.plot(kind='barh', stacked=True, )
plt.xlabel('Count of Passes')
plt.ylabel('Taxonomic Level')
plt.title('Pipeline Passes by Attempted Taxonomy')

plt.show()

In [ ]:
# Total alignment

import plotly.graph_objects as go

tax_map = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]

columns_to_include = ["gpt_taxonomy", "classified_taxonomy"]
selected_df = df[columns_to_include]


d = {}
for index, h in selected_df.iterrows():
  gpt, ml = h["gpt_taxonomy"], h["classified_taxonomy"]
  k = (tax_map.index(gpt), tax_map.index(ml)+6)
  if k in d:
    d[k] += 1
  else:
    d[k] = 1

color_map = {
    "Remember": "orange",
    "Understand": "blue",
    "Apply": "green",
    "Analyze": "yellow",
    "Evaluate": "red",
    "Create": "pink"
}

source = []
target = []
values = []
colors = []

for key, value in d.items():
  source.append(key[0])
  target.append(key[1])
  values.append(value)
  colors.append(color_map[tax_map[key[0]]])


fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create", "Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"],
      color = "orange"
    ),
    link = dict(
      source = source, # indices correspond to labels, eg A1, A2, A1, B1, ...
      target = target,
      value = values,
      color = colors
  ))])

fig.update_layout(title_text="Alignment Between Targeted and True Bloom Level",
                  font_size=15,
                  width=800)
fig.show()

In [ ]:
import matplotlib.pyplot as plt

columns_to_include = ["ambiguous_unclear_information",
                      "implausible_distractors",
                      "none_of_the_above",
                      "longest_answer_correct",
                      "gratuitous_information_in_stem",
                      "true_or_false",
                      "avoid_convergence_cues",
                      "avoid_logical_cues",
                      "all_of_the_above",
                      "fill_in_the_blank",
                      'absolute_terms',
                      'word_repeats_in_stem_and_correct_answer',
                      "unfocused_stem",
                      "complex_k_type",
                      "grammatical_cues_in_stem",
                      "lost_sequence",
                      "vague_terms",
                      "negative_worded_stem",
                      "more_than_one_correct"]

names_to_labels = {"ambiguous_unclear_information": "Ambiguous/Unclear Info",
                      "implausible_distractors": "Implausible Distractors",
                      "none_of_the_above": "Has None of the Above",
                      "longest_answer_correct": "Longest Answer Correct",
                      "gratuitous_information_in_stem": "Gratuitous Info in Stem",
                      "true_or_false": "Is a True/False Question",
                      "avoid_convergence_cues": "Convergence Cues",
                      "avoid_logical_cues": "Logical Cues",
                      "all_of_the_above": "Has All of the Above",
                      "fill_in_the_blank": "Fill in the Blank",
                      'absolute_terms': "Absolute Terms",
                      'word_repeats_in_stem_and_correct_answer': "Repeated Words",
                      "unfocused_stem": "Unfocused Stem",
                      "complex_k_type": "Complex K Type",
                      "grammatical_cues_in_stem": "Grammatical Cues in Stem",
                      "lost_sequence": "Lost Sequence",
                      "vague_terms": "Vague Terms",
                      "negative_worded_stem": "Negatively Worded Stem",
                      "more_than_one_correct": "More than One Option Correct"}

# Filter the DataFrame to include only the selected columns
selected_df = df[columns_to_include]
selected_df = selected_df.rename(columns=names_to_labels)

# Calculate the count of True values for each column
false_counts = ((~selected_df).sum()/250 *100).sort_values()
# Create a histogram
false_counts.plot(kind='barh')
plt.xlim(0, 100)
plt.xlabel('Percent of Fails')
plt.ylabel('IWF Criteria')
plt.title('Percent of Fails by IWF Criteria')

plt.show()
print(false_counts)

In [ ]:
columns_to_include = ["gpt_taxonomy",
                      "ambiguous_unclear_information",
                      "implausible_distractors",
                      "none_of_the_above",
                      "longest_answer_correct",
                      "gratuitous_information_in_stem",
                      "true_or_false",
                      "avoid_convergence_cues",
                      "avoid_logical_cues",
                      "all_of_the_above",
                      "fill_in_the_blank",
                      'absolute_terms',
                      'word_repeats_in_stem_and_correct_answer',
                      "unfocused_stem",
                      "complex_k_type",
                      "grammatical_cues_in_stem",
                      "lost_sequence",
                      "vague_terms",
                      "negative_worded_stem",
                      "more_than_one_correct"]

# Filter the DataFrame to include only the selected columns
selected_df = df[columns_to_include]

selected_df['total_true'] = selected_df.drop(columns=['gpt_taxonomy']).sum(axis=1)

mean_total_true_by_class = selected_df.groupby('gpt_taxonomy')['total_true'].mean()

# Create a histogram
mean_total_true_by_class.plot(kind='bar')
plt.xlabel('Attempted Taxonomy')
plt.ylabel('Average IWF Passes')
plt.title('Average IWF Passes by Attempted Taxonomy')

plt.show()

In [ ]:
columns_to_include = ["shot_num",
                      "ambiguous_unclear_information",
                      "implausible_distractors",
                      "none_of_the_above",
                      "longest_answer_correct",
                      "gratuitous_information_in_stem",
                      "true_or_false",
                      "avoid_convergence_cues",
                      "avoid_logical_cues",
                      "all_of_the_above",
                      "fill_in_the_blank",
                      'absolute_terms',
                      'word_repeats_in_stem_and_correct_answer',
                      "unfocused_stem",
                      "complex_k_type",
                      "grammatical_cues_in_stem",
                      "lost_sequence",
                      "vague_terms",
                      "negative_worded_stem",
                      "more_than_one_correct"]

# Filter the DataFrame to include only the selected columns
selected_df = df[columns_to_include]

selected_df['total_true'] = selected_df.drop(columns=['shot_num']).sum(axis=1)

mean_total_true_by_class = selected_df.groupby('shot_num')['total_true'].mean()

# Create a histogram
mean_total_true_by_class.plot(kind='bar')
plt.xlabel('Shot Number')
plt.ylabel('Average IWF Passes')
plt.title('Average IWF Passes by Shot Number')

plt.show()